# Stage 0: old C1 with two seeds, and one update per seed (350-scene subset)

First run of the next stage (Meeting 6 report, Section 4.1).

1. Train the old C1 (YOLO11n) on `boston_c1` (100 scenes) with seeds 0 and 1.
2. Update each old C1 on `singapore_update` (100 scenes) with the same seed.
3. Evaluate all four models on `singapore_test` (50 scenes, 2,020 images).
4. Write the detections of all four models on `singapore_test` (no ground truth needed) for the H2 measures.
5. Save every model to `/kaggle/working/models/` with `manifest.json` (config, metrics, sha256) for reuse.

Model selection uses `last.pt`. Training runs with `val=False`, so the test scenes are never used to pick a checkpoint.
Answers: is the gap between the two old C1 seeds (random variation) small compared with the effect of an update, and is 350 scenes enough?

In [ ]:
import os, sys, glob, json, time, tarfile, hashlib, shutil, subprocess
from pathlib import Path
T0 = time.time()
def clock(m): print(f'[{time.time()-T0:7.1f}s] {m}', flush=True)

# --- labsd package (from ifty1011/labsd-src) ---
LABSD_DIR = '/kaggle/working/labsd_pkg'; Path(LABSD_DIR).mkdir(parents=True, exist_ok=True)
def locate_labsd():
    for init in glob.glob('/kaggle/input/**/labsd/__init__.py', recursive=True):
        return str(Path(init).parent.parent)
    for tar in glob.glob('/kaggle/input/**/labsd.tar', recursive=True):
        with tarfile.open(tar) as tf: tf.extractall(LABSD_DIR)
        for init in glob.glob(LABSD_DIR + '/**/labsd/__init__.py', recursive=True):
            return str(Path(init).parent.parent)
P = locate_labsd(); assert P, 'labsd not found'
if P not in sys.path: sys.path.insert(0, P)
import labsd
clock('labsd: ' + labsd.__file__)

# --- 350-scene subset (output of ifty1011/labsd-e1-subset350) ---
SPLITS = glob.glob('/kaggle/input/**/splits_350.json', recursive=True)
assert SPLITS, 'splits_350.json not found: attach kernel ifty1011/labsd-e1-subset350'
NUSC_ROOT = str(Path(SPLITS[0]).parent)
assert Path(NUSC_ROOT, 'v1.0-trainval', 'scene.json').exists()
splits = json.load(open(SPLITS[0]))
clock(f'subset root: {NUSC_ROOT}  parts: ' + str({k: len(v) for k, v in splits.items()}))

In [ ]:
clock('installing deps...')
pip = [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q']
subprocess.run(pip + ['--no-deps', 'nuscenes-devkit'], check=True)
subprocess.run(pip + ['pyquaternion', 'cachetools', 'descartes', 'shapely'], check=True)
subprocess.run(pip + ['--no-deps', 'ultralytics==8.4.105'], check=True)
subprocess.run(pip + ['ultralytics-thop', 'py-cpuinfo'], check=True)
import torch
assert torch.cuda.is_available(), 'no GPU: enable a GPU accelerator'
cap = torch.cuda.get_device_capability(0)
clock(f'GPU {torch.cuda.get_device_name(0)} sm_{cap[0]}{cap[1]}')
assert cap[0] >= 7, 'GPU below sm_70 (P100): select T4 so training does not fall back to CPU'
# The subset has no maps/ folder, and the devkit checks every map image at start-up.
# Maps are not used here: build a working root that links the subset's metadata and
# images, and points every map record at a 1x1 placeholder image (tested locally).
from PIL import Image
WORK_ROOT = Path('/kaggle/working/nusc_root')
(WORK_ROOT / 'v1.0-trainval').mkdir(parents=True, exist_ok=True)
(WORK_ROOT / 'maps').mkdir(exist_ok=True)
Image.new('L', (1, 1)).save(WORK_ROOT / 'maps' / 'placeholder.png')
if not (WORK_ROOT / 'samples').exists(): os.symlink(Path(NUSC_ROOT) / 'samples', WORK_ROOT / 'samples')
for f in Path(NUSC_ROOT, 'v1.0-trainval').glob('*.json'):
    dst = WORK_ROOT / 'v1.0-trainval' / f.name
    if dst.exists() or dst.is_symlink(): continue
    if f.name == 'map.json':
        recs = json.load(open(f))
        for r in recs: r['filename'] = 'maps/placeholder.png'
        dst.write_text(json.dumps(recs))
    else: os.symlink(f, dst)
from nuscenes.nuscenes import NuScenes
NUSC = NuScenes(version='v1.0-trainval', dataroot=str(WORK_ROOT), verbose=False)
clock(f'NuScenes loaded: {len(NUSC.scene)} scenes in metadata')

In [ ]:
from labsd.c1_yolo import build_yolo_dataset, write_yolo_data_yaml
def make_ds(name, train_part):
    d = f'/kaggle/working/ds_{name}'
    n_tr = build_yolo_dataset(NUSC, splits[train_part], d, split_name='train')
    n_te = build_yolo_dataset(NUSC, splits['singapore_test'], d, split_name='val')
    clock(f'{name}: train images with labels={n_tr}, test images with labels={n_te}')
    return write_yolo_data_yaml(d), n_tr, n_te
YAML_BOS, N_BOS, N_TEST = make_ds('boston_c1', 'boston_c1')
YAML_SG,  N_SG,  _      = make_ds('singapore_update', 'singapore_update')

In [ ]:
from ultralytics import YOLO
MODELS = Path('/kaggle/working/models'); MODELS.mkdir(exist_ok=True)
CFG = dict(epochs=10, imgsz=640, batch=8, base='yolo11n.pt')   # same settings as the earlier experiments
SEEDS = [0, 1]

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()

def train(name, data_yaml, base_weights, seed):
    out = MODELS / f'{name}.pt'
    if out.exists():
        clock(f'{name}: already saved, skipping'); return str(out)
    t = time.time()
    YOLO(base_weights).train(data=data_yaml, epochs=CFG['epochs'], imgsz=CFG['imgsz'],
                             batch=CFG['batch'], device='0', seed=seed, val=False,
                             project='/kaggle/working/runs', name=name, exist_ok=True,
                             verbose=False, plots=False)
    shutil.copy2(f'/kaggle/working/runs/{name}/weights/last.pt', out)
    clock(f'{name}: trained in {time.time()-t:.0f}s -> {out}')
    return str(out)

def evaluate(weights):
    m = YOLO(weights).val(data=YAML_BOS, split='val', imgsz=CFG['imgsz'], batch=16,
                          device='0', verbose=False, plots=False)
    return {'mAP50': float(m.box.map50), 'mAP50_95': float(m.box.map),
            'precision': float(m.box.mp), 'recall': float(m.box.mr)}

manifest = {'config': CFG, 'subset': 'ifty1011/labsd-e1-subset350',
            'train_images': {'boston_c1': N_BOS, 'singapore_update': N_SG},
            'test_images_with_labels': N_TEST, 'checkpoint': 'last.pt (no test-set selection)',
            'models': {}}
def record(name, path, parent, seed, part):
    manifest['models'][name] = {'file': f'{name}.pt', 'sha256': sha256(path),
                                'init': parent, 'seed': seed, 'trained_on': part}
    json.dump(manifest, open(MODELS / 'manifest.json', 'w'), indent=1)

for s in SEEDS:
    p_old = train(f'c1_old_s{s}', YAML_BOS, CFG['base'], s)
    record(f'c1_old_s{s}', p_old, CFG['base'], s, 'boston_c1')
    p_new = train(f'c1_new_s{s}', YAML_SG, p_old, s)
    record(f'c1_new_s{s}', p_new, f'c1_old_s{s}', s, 'singapore_update')
clock('all four models trained and saved')

In [ ]:
# Accuracy on the 50 test scenes (uses ground truth; for delta_1 and the seed check)
for name, info in manifest['models'].items():
    info['test_metrics'] = evaluate(str(MODELS / info['file']))
    clock(f"{name}: " + json.dumps(info['test_metrics']))
json.dump(manifest, open(MODELS / 'manifest.json', 'w'), indent=1)

d1 = {s: manifest['models'][f'c1_new_s{s}']['test_metrics']['mAP50']
         - manifest['models'][f'c1_old_s{s}']['test_metrics']['mAP50'] for s in SEEDS}
seed_gap = abs(manifest['models']['c1_old_s0']['test_metrics']['mAP50']
               - manifest['models']['c1_old_s1']['test_metrics']['mAP50'])
summary = {'delta1_mAP50_per_seed': d1, 'old_c1_seed_gap_mAP50': seed_gap,
           'paper_old_c1_mAP50_280_scenes': 0.1370}
print(json.dumps(summary, indent=1))

In [ ]:
# Detections on the 50 test scenes, no ground truth needed (input to the H2 measures)
from labsd.c1_yolo import dump_detections
DET = Path('/kaggle/working/detections'); DET.mkdir(exist_ok=True)
for name, info in manifest['models'].items():
    t = time.time()
    recs = dump_detections(str(MODELS / info['file']), splits['singapore_test'], NUSC, score_thresh=0.25)
    json.dump(recs, open(DET / f'{name}_singapore_test.json', 'w'))
    info['n_detections_test'] = len(recs)
    clock(f'{name}: {len(recs)} detections in {time.time()-t:.0f}s')
json.dump(manifest, open(MODELS / 'manifest.json', 'w'), indent=1)
summary['n_detections_test'] = {k: v['n_detections_test'] for k, v in manifest['models'].items()}
json.dump(summary, open('/kaggle/working/stage0_summary.json', 'w'), indent=1)
shutil.rmtree('/kaggle/working/runs', ignore_errors=True)
for d in glob.glob('/kaggle/working/ds_*'): shutil.rmtree(d, ignore_errors=True)
print(json.dumps(summary, indent=1))
clock('=== STAGE 0 DONE ===')